In [1]:
# MLIR Based on the following: https://openxla.org/stablehlo/tutorials/jax-export
#!pip install -U jax jaxlib flax transformers tf-nightly
import jax
from jax import export
import jax.numpy as jnp
import numpy as np
from jax._src.interpreters import mlir as jax_mlir
from jax._src.lib.mlir import ir
from jax._src.lib.mlir import passmanager as pm

# stablehlo-opt -stablehlo-legalize-to-linalg <filename.mlir>

# Returns prettyprint of StableHLO module without large constants
def get_stablehlo_asm(module_str):
  with jax_mlir.make_ir_context():
    stablehlo_module = ir.Module.parse(module_str, context=jax_mlir.make_ir_context())
    return stablehlo_module.operation.get_asm(large_elements_limit=20)

# Disable logging for better tutorial rendering
import logging
logging.disable(logging.WARNING)

# crearing GEMM gemm add
@jax.jit
def gemm_add(a, b, d):
  return jnp.add(jnp.matmul(a, b), d)
inputs = (
    np.ones((10, 10), dtype=np.float32),
    np.ones((10, 10), dtype=np.float32),
    np.ones((10, 10), dtype=np.float32),
)
input_shapes = [jax.ShapeDtypeStruct(x.shape, x.dtype) for x in inputs]
stablehlo_gemm = export.export(gemm_add)(*input_shapes).mlir_module()
print(get_stablehlo_asm(stablehlo_gemm))

module @jit_gemm_add attributes {jax.uses_shape_polymorphism = false, mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<10x10xf32>, %arg1: tensor<10x10xf32>, %arg2: tensor<10x10xf32>) -> (tensor<10x10xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [1] x [0] : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %1 = stablehlo.add %0, %arg2 : tensor<10x10xf32>
    return %1 : tensor<10x10xf32>
  }
}



# Compiling using JAX rocMLIR

Starting with the output from above:

```
module @jit_gemm_add attributes {jax.uses_shape_polymorphism = false, mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
  func.func public @main(%arg0: tensor<10x10xf32>, %arg1: tensor<10x10xf32>, %arg2: tensor<10x10xf32>) -> (tensor<10x10xf32> {jax.result_info = "result"}) {
    %0 = stablehlo.dot_general %arg0, %arg1, contracting_dims = [1] x [0] : (tensor<10x10xf32>, tensor<10x10xf32>) -> tensor<10x10xf32>
    %1 = stablehlo.add %0, %arg2 : tensor<10x10xf32>
    return %1 : tensor<10x10xf32>
  }
}
```

Run the following command to go from StableHLO to Rock Dialect

```
./bin/stablehlo-opt  --stablehlo-legalize-to-linalg=enable-primitive-ops |\
      /home/vhe/rocMLIR/build-release/bin/rocmlir-driver --kernel-pipeline=highlevel
```

Gives the following rock output:

```
map = affine_map<(d0, d1) -> (d0, d1)>
module @jit_gemm_add attributes {jax.uses_shape_polymorphism = false, mhlo.num_partitions = 1 : i32, mhlo.num_replicas = 1 : i32} {
func.func public @main(%arg0: memref<10x10xf32>, %arg1: memref<10x10xf32>, %arg2: memref<10x10xf32>, %arg3: memref<10x10xf32> {jax.result_info = "result"}) attributes {arch = "gfx950", kernel} {
  %alloc = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
    rock.gemm %alloc = %arg0 * %arg1 storeMethod =  set : memref<10x10xf32> = memref<10x10xf32> * memref<10x10xf32>
    %alloc_0 = memref.alloc() {alignment = 64 : i64} : memref<10x10xf32>
    linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel", "parallel"]} ins(%alloc, %arg2 : memref<10x10xf32>, memref<10x10xf32>) outs(%alloc_0 : memref<10x10xf32>) {
      ^bb0(%in: f32, %in_1: f32, %out: f32):
        %0 = arith.addf %in, %in_1 : f32
         linalg.yield %0 : f32
    }
  memref.copy %alloc_0, %arg3 : memref<10x10xf32> to memref<10x10xf32>
    return
}
```